# Performance vs. annotation budget

The first three graphs show one mean curve per configuration, grouping runs that differ only in name and seed. Scores and actual frame budgets are averaged across seeds at the same acquisition round. Shaded bands show ±1 sample standard deviation (`ddof=1`) of the scores; no band is estimated for rounds with only one seed. Legends report the number of contributing seeds (or its range across rounds). Seed counts can decrease at later rounds while training is incomplete.

The last six graphs show individual mean-pooling runs separately for seeds 100 and 200 for each of the three metrics. Random sampling is shown with dashed black lines throughout.

Budget is the percentage of training frames actually labeled. Each round uses the epoch with the highest validation edit score, matching checkpoint selection. Incomplete rounds are excluded. Lines connect observations without extrapolation.

Re-run all cells to refresh the graphs as training progresses.

In [ ]:
# If needed in this kernel: %pip install numpy pandas matplotlib nbformat ipykernel
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
pd.set_option("display.max_colwidth", 100)

# Works from the repository root, this notebook's directory, or a subdirectory.
relative_runs = Path("experiments/4_active_learning_runs")
RUNS_DIR = next(
    (base / relative_runs for base in [Path.cwd(), *Path.cwd().parents]
     if (base / relative_runs).is_dir()), None
)
if RUNS_DIR is None:
    raise FileNotFoundError("Run from inside the alpes repository or set RUNS_DIR explicitly.")
METRICS = {"val_edit": "Validation edit score", "val_f1_element": "Validation F1 element",
           "val_f1_event": "Validation F1 event"}
print(f"Loading runs from {RUNS_DIR}")

In [ ]:
def read_json(path):
    with path.open() as stream:
        return json.load(stream)


records, inventory, skipped = [], [], []
settings_by_group = {}
for config_path in sorted(RUNS_DIR.glob("*/config.json")):
    config = read_json(config_path)
    if config.get("query_score_pooling", "").upper() != "MEAN":
        continue
    run_dir = config_path.parent
    assert config["active_learning_budget_unit"] == "percent_of_training_frames"
    total_frames = config["total_training_frames"]
    assert total_frames > 0
    settings = {k: v for k, v in config.items() if k not in {"name", "seed"}}
    group = json.dumps(settings, sort_keys=True)
    settings_by_group[group] = settings
    accepted = 0
    for round_dir in sorted(run_dir.glob("round_*")):
        loss_path, labeled_path = round_dir / "loss.json", round_dir / "labeled_train.json"
        reason = None
        if not loss_path.exists() or not labeled_path.exists():
            reason = "Missing loss or labeled-pool file"
        else:
            losses = pd.DataFrame(read_json(loss_path))
            expected_epochs = config["f3ed"]["num_epochs"]
            if "epoch" not in losses or not set(range(expected_epochs)).issubset(set(losses["epoch"])):
                reason = "Training incomplete"
            elif not set(METRICS).issubset(losses.columns):
                reason = "Missing validation metrics"
            else:
                evaluated = losses.loc[
                    losses["epoch"] >= config["f3ed"]["start_val_epoch"]
                ].dropna(subset=list(METRICS)).sort_values("epoch")
                if evaluated.empty:
                    reason = "No evaluated epoch"
        if reason:
            skipped.append({"run": run_dir.name, "round": round_dir.name, "reason": reason})
            continue
        best = evaluated.loc[evaluated["val_edit"].idxmax()]
        labeled_frames = sum(item["num_frames"] for item in read_json(labeled_path))
        assert 0 < labeled_frames <= total_frames
        assert np.isfinite(best[list(METRICS)].to_numpy(dtype=float)).all()
        records.append({"run": run_dir.name, "group": group, "seed": config["seed"],
                        "strategy": config["query_strategy"],
                        "round": int(round_dir.name.split("_")[-1]),
                        "labeled_frames": labeled_frames,
                        "budget_pct": 100 * labeled_frames / total_frames,
                        "best_epoch": int(best["epoch"]),
                        **{metric: float(best[metric]) for metric in METRICS}})
        accepted += 1
    inventory.append({"run": run_dir.name, "group": group, "seed": config["seed"],
                      "completed_rounds": accepted})

rounds = pd.DataFrame(records)
if rounds.empty:
    raise ValueError("No completed mean-pooling rounds found.")
# Keep labels readable, but distinguish any additional configurations of a strategy.
labels = {}
for group, settings in settings_by_group.items():
    strategy = settings["query_strategy"]
    label = strategy.replace("_MEASURE", "").replace("_", " ").title()
    if strategy == "RANDOM_SAMPLING":
        label = "Random sampling (baseline)"
    peers = [g for g, s in settings_by_group.items() if s["query_strategy"] == strategy]
    if len(peers) > 1:
        label += f" [config {peers.index(group) + 1}]"
    labels[group] = label
rounds["setting"] = rounds["group"].map(labels)
inventory = pd.DataFrame(inventory)
inventory["setting"] = inventory["group"].map(labels)
assert not inventory.duplicated(["group", "seed"]).any(), "Duplicate seed within settings"
for _, curve in rounds.groupby("run"):
    assert np.all(np.diff(curve.sort_values("round")["budget_pct"]) > 0)

print(f"{len(inventory)} mean-pooling runs; {len(rounds)} completed rounds.")
display(inventory[["setting", "seed", "completed_rounds", "run"]])
if skipped:
    print("Excluded rounds (re-run after training completes):")
    display(pd.DataFrame(skipped))

In [ ]:
group_order = sorted(rounds["group"].unique(), key=lambda g: (
    settings_by_group[g]["query_strategy"] != "RANDOM_SAMPLING", labels[g]))
strategy_groups = [g for g in group_order
                   if settings_by_group[g]["query_strategy"] != "RANDOM_SAMPLING"]
colors = {g: plt.get_cmap("tab10")(i % 10) for i, g in enumerate(strategy_groups)}
seeds = sorted(rounds["seed"].unique())
markers = {seed: ["o", "s", "^", "D", "v", "P", "X"][i % 7]
           for i, seed in enumerate(seeds)}


def plot_metric(metric, seed=None, ylim=None):
    data = rounds if seed is None else rounds.loc[rounds["seed"] == seed]
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for group in group_order:
        baseline = settings_by_group[group]["query_strategy"] == "RANDOM_SAMPLING"
        group_data = data.loc[data["group"] == group]
        color = "black" if baseline else colors[group]
        linestyle = "--" if baseline else "-"
        if seed is None:
            summary = group_data.groupby("round", sort=True).agg(
                budget_pct=("budget_pct", "mean"),
                mean=(metric, "mean"), std=(metric, "std"),
                n_seeds=("seed", "nunique"),
            )
            n_min, n_max = summary["n_seeds"].min(), summary["n_seeds"].max()
            count = str(n_min) if n_min == n_max else f"{n_min}–{n_max}"
            ax.plot(summary["budget_pct"], summary["mean"],
                    color=color, linestyle=linestyle, marker="o",
                    markersize=4, linewidth=1.8,
                    label=f"{labels[group]} (n={count})")
            ax.fill_between(
                summary["budget_pct"].to_numpy(),
                (summary["mean"] - summary["std"]).to_numpy(),
                (summary["mean"] + summary["std"]).to_numpy(),
                color=color, alpha=0.15,
            )
        else:
            for _, curve in group_data.groupby("run", sort=True):
                curve = curve.sort_values("budget_pct")
                ax.plot(
                    curve["budget_pct"], curve[metric],
                    color=color, linestyle=linestyle, marker=markers[seed],
                    markersize=4, linewidth=1.8,
                    label=f"{labels[group]} — seed {seed}",
                )
    ax.set(xlabel="Annotation budget (% of training frames)",
           ylabel=METRICS[metric], title=f"{METRICS[metric]} vs. annotation budget"
                 + (f" — seed {seed}" if seed is not None else " — mean ± 1 SD"),
           xlim=(0, 101))
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xticks(np.arange(0, 101, 10))
    ax.xaxis.set_major_formatter(PercentFormatter(100))
    ax.grid(alpha=0.2)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    plt.show()


plot_metric("val_edit")

In [ ]:
plot_metric("val_f1_element")
plot_metric("val_f1_event")

## Comparison by seed

Validation edit score, F1 element, and F1 event for all mean-pooling runs with seed 100 and seed 200, shown separately with matching colors and axis limits within each metric pair. Each graph includes all available strategies for that seed, using completed rounds only.

In [ ]:
def plot_seed_pair(metric):
    paired_scores = rounds.loc[rounds["seed"].isin([100, 200,]), metric]
    min_padding = 1.0 if metric == "val_edit" else 0.01
    padding = max((paired_scores.max() - paired_scores.min()) * 0.05, min_padding)
    paired_ylim = (paired_scores.min() - padding, paired_scores.max() + padding)
    for seed in (100, 200, 300):
        plot_metric(metric, seed=seed, ylim=paired_ylim)


plot_seed_pair("val_edit")

In [ ]:
plot_seed_pair("val_f1_element")

In [ ]:
plot_seed_pair("val_f1_event")